**Install Packages**

In [1]:
!pip -q install kagglehub timm captum lime scikit-learn scikit-image opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 11.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.3 MB/s eta 0:00:00


**Imports**

In [2]:
import json, warnings
from pathlib import Path

import cv2
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix

from torch.utils.data import Dataset, DataLoader

from captum.attr import IntegratedGradients, Saliency, NoiseTunnel
from lime import lime_image
from skimage.segmentation import mark_boundaries

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

**Project Folder Names**

In [4]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_FOLDER = "BRSET XAI Project"

PROCESSED_SHARED_FOLDER = "processed shared"
PREPROCESSED_DATASET_FOLDER = "preprocessed dataset"
MODEL_OUTPUTS_FOLDER = "model outputs"

MODEL_FOLDER = "ViT RETFound-Style"

IMAGE_MANIFEST_FILENAME = "brset image manifest.csv"
PREPROCESSING_METADATA_FILENAME = "preprocessing metadata.json"

BEST_MODEL_FILENAME = "best.pt"
LAST_MODEL_FILENAME = "last.pt"
HISTORY_FILENAME = "history.csv"
TEST_PROBS_FILENAME = "test probabilities.npy"
METRICS_FILENAME = "metrics.csv"
CLASSIFICATION_REPORT_FILENAME = "classification report.csv"
CONFUSION_MATRIX_CSV_FILENAME = "confusion matrix.csv"
CONFUSION_MATRIX_FIGURE_FILENAME = "confusion matrix.png"
TRAINING_CURVES_FIGURE_FILENAME = "training curves.png"
OCCLUSION_SENSITIVITY_FILENAME = "occlusion sensitivity.png"
INTEGRATED_GRADIENTS_FILENAME = "integrated gradients.png"
SMOOTHGRAD_FILENAME = "smoothgrad.png"
LIME_FILENAME = "lime.png"

DATASET_SLUG = "tanzinabdul/fundus-patientwise-split"

Mounted at /content/drive


**Derived Paths**

In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_FOLDER

PROCESSED_SHARED_DIR = PROJECT_ROOT / PROCESSED_SHARED_FOLDER
PREPROCESSED_DATASET_DIR = PROCESSED_SHARED_DIR / PREPROCESSED_DATASET_FOLDER

MODEL_OUTPUTS_DIR = PROJECT_ROOT / MODEL_OUTPUTS_FOLDER
CURRENT_MODEL_OUTPUT_DIR = MODEL_OUTPUTS_DIR / MODEL_FOLDER

IMAGE_MANIFEST_PATH = PREPROCESSED_DATASET_DIR / IMAGE_MANIFEST_FILENAME
PREPROCESSING_METADATA_PATH = PREPROCESSED_DATASET_DIR / PREPROCESSING_METADATA_FILENAME

BEST_MODEL_PATH = CURRENT_MODEL_OUTPUT_DIR / BEST_MODEL_FILENAME
LAST_MODEL_PATH = CURRENT_MODEL_OUTPUT_DIR / LAST_MODEL_FILENAME
HISTORY_PATH = CURRENT_MODEL_OUTPUT_DIR / HISTORY_FILENAME
TEST_PROBS_PATH = CURRENT_MODEL_OUTPUT_DIR / TEST_PROBS_FILENAME
METRICS_PATH = CURRENT_MODEL_OUTPUT_DIR / METRICS_FILENAME
CLASSIFICATION_REPORT_PATH = CURRENT_MODEL_OUTPUT_DIR / CLASSIFICATION_REPORT_FILENAME
CONFUSION_MATRIX_CSV_PATH = CURRENT_MODEL_OUTPUT_DIR / CONFUSION_MATRIX_CSV_FILENAME
CONFUSION_MATRIX_FIGURE_PATH = CURRENT_MODEL_OUTPUT_DIR / CONFUSION_MATRIX_FIGURE_FILENAME
TRAINING_CURVES_FIGURE_PATH = CURRENT_MODEL_OUTPUT_DIR / TRAINING_CURVES_FIGURE_FILENAME
OCCLUSION_SENSITIVITY_PATH = CURRENT_MODEL_OUTPUT_DIR / OCCLUSION_SENSITIVITY_FILENAME
INTEGRATED_GRADIENTS_PATH = CURRENT_MODEL_OUTPUT_DIR / INTEGRATED_GRADIENTS_FILENAME
SMOOTHGRAD_PATH = CURRENT_MODEL_OUTPUT_DIR / SMOOTHGRAD_FILENAME
LIME_PATH = CURRENT_MODEL_OUTPUT_DIR / LIME_FILENAME

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREPROCESSED_DATASET_DIR:", PREPROCESSED_DATASET_DIR)
print("CURRENT_MODEL_OUTPUT_DIR:", CURRENT_MODEL_OUTPUT_DIR)

PROJECT_ROOT: /content/drive/MyDrive/BRSET XAI Project
PREPROCESSED_DATASET_DIR: /content/drive/MyDrive/BRSET XAI Project/processed shared/preprocessed dataset
CURRENT_MODEL_OUTPUT_DIR: /content/drive/MyDrive/BRSET XAI Project/model outputs/ViT RETFound-Style


**Kaggle Download**

In [6]:
import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(DATASET_SLUG)

RAW_DIR = Path(dataset_path)

print("Dataset downloaded successfully.")
print("RAW_DIR:", RAW_DIR)

100%|██████████| 12.2G/12.2G [01:58<00:00, 111MB/s]

Extracting files...


Dataset downloaded successfully.
RAW_DIR: /root/.cache/kagglehub/datasets/tanzinabdul/fundus-patientwise-split/versions/1


**Model Config**

In [7]:
import sys
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

In [9]:
from funduspreprocessing import preprocess_fundus
from model_common import (
    SEED,
    IMG_SIZE,
    BATCH_SIZE,
    EPOCHS,
    LR,
    WEIGHT_DECAY,
    NUM_WORKERS,
    ROTATION_DEGREES,
    set_seed,
    build_image_transforms,
    denormalize_image_tensor,
    predict_model,
    evaluate_predictions,
    save_metrics,
    save_probabilities,
    save_classification_report,
    save_confusion_matrix_csv,
    train_with_validation,
)

set_seed(SEED)

preprocessing_metadata = json.load(open(PREPROCESSING_METADATA_PATH))
TARGET_COL = preprocessing_metadata["target"]["target_column"]

NUM_WORKERS = 0

**Load Processed Data**

In [10]:
df = pd.read_csv(IMAGE_MANIFEST_PATH)

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]
image_files = [
    path
    for path in RAW_DIR.rglob("*")
    if path.suffix.lower() in IMAGE_EXTENSIONS
]
image_path_map = {
    path.stem: str(path)
    for path in image_files
}

df["processed_image_path"] = df["processed_image_path"].apply(
    lambda x: image_path_map.get(Path(str(x)).stem, str(x))
)

df[TARGET_COL] = df[TARGET_COL].astype(int)

display(df.head())
print(df.shape)
print(df["split"].value_counts())
print(df[TARGET_COL].value_counts().sort_index())

,processed_image_path,target,split
0,/root/.cache/kagglehub/datasets/tanzinabdul/fu...,0,train
1,/root/.cache/kagglehub/datasets/tanzinabdul/fu...,0,train
2,/root/.cache/kagglehub/datasets/tanzinabdul/fu...,0,train
3,/root/.cache/kagglehub/datasets/tanzinabdul/fu...,0,train
4,/root/.cache/kagglehub/datasets/tanzinabdul/fu...,0,train


(16258, 3)
split
train    13007
val       1632
test      1619
Name: count, dtype: int64
target
0    15180
1      157
2      449
3       77
4      395
Name: count, dtype: int64


**Dataset Class**

In [11]:
train_tfms, test_tfms = build_image_transforms(
    rotation_degrees=ROTATION_DEGREES
)

class BRSETDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img = preprocess_fundus(
            row["processed_image_path"],
            img_size=IMG_SIZE
        )

        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        y = torch.tensor(int(row[TARGET_COL]), dtype=torch.long)

        return img, y

**DataLoaders**

In [12]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

num_classes = int(df[TARGET_COL].max() + 1)

train_ds = BRSETDataset(train_df, transform=train_tfms)
val_ds = BRSETDataset(val_df, transform=test_tfms)
test_ds = BRSETDataset(test_df, transform=test_tfms)

pin_memory = DEVICE.type == "cuda"

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_df[TARGET_COL].to_numpy()
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Classes:", num_classes)
print("Class weights:", class_weights)

Classes: 5
Class weights: tensor([ 0.2141, 21.1496,  7.3279, 40.0215,  8.2323])


**ViT Batch Loading Timing Test**

In [ ]:
import time

start = time.time()

debug_batch = next(iter(train_loader))

debug_imgs, debug_labels = debug_batch

elapsed = time.time() - start

print("One ViT batch loaded successfully.")
print("Image batch shape:", debug_imgs.shape)
print("Label batch shape:", debug_labels.shape)
print("Time to load one batch:", round(elapsed, 2), "seconds")

**ViT RETFound-Style Model**

In [13]:
class ViTImageModel(nn.Module):
    def __init__(self, backbone_name="vit_base_patch16_224", num_classes=5):
        super().__init__()

        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)

vit_model = ViTImageModel(
    "vit_base_patch16_224",
    num_classes
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    vit_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

**ViT Forward Timing Test**

In [ ]:
start = time.time()

vit_model.eval()

debug_imgs = debug_imgs.to(DEVICE)

with torch.no_grad():
    debug_logits = vit_model(debug_imgs)

elapsed = time.time() - start

print("One ViT forward pass completed.")
print("Logits shape:", debug_logits.shape)
print("Time for one forward pass:", round(elapsed, 2), "seconds")

**Train Model**

In [ ]:
RESUME_TRAINING = False

history, best_macro_f1 = train_with_validation(
    model=vit_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    epochs=EPOCHS,
    best_model_path=BEST_MODEL_PATH,
    last_model_path=LAST_MODEL_PATH,
    history_path=HISTORY_PATH,
    resume_training=RESUME_TRAINING,
    monitor_metric="macro_f1"
)

Starting training from scratch


**Training Curves**

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["epoch"], history_df["loss"], marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Macro F1")
axes[1].plot(history_df["epoch"], history_df["val_bal_acc"], marker="o", label="Balanced Accuracy")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()

plt.tight_layout()
plt.savefig(TRAINING_CURVES_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

**Test Model**

In [ ]:
vit_model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=DEVICE)
)

y_true, y_pred, y_prob, infer_time = predict_model(
    vit_model,
    test_loader,
    DEVICE
)

vit_metrics = evaluate_predictions(
    y_true=y_true,
    y_pred=y_pred,
    y_prob=y_prob,
    model_name="ViT RETFound-Style",
    inference_time_seconds=infer_time
)

save_probabilities(y_prob, TEST_PROBS_PATH)
save_metrics(vit_metrics, METRICS_PATH)

save_classification_report(
    y_true,
    y_pred,
    CLASSIFICATION_REPORT_PATH
)

save_confusion_matrix_csv(
    y_true,
    y_pred,
    CONFUSION_MATRIX_CSV_PATH
)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("ViT RETFound-Style Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_FIGURE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(vit_metrics)

**XAI Helper: Denormalization**

In [ ]:
xai_sample_index = int(np.argmax(y_prob.max(axis=1)))
sample_img, sample_y = test_ds[xai_sample_index]

img_np = denormalize_image_tensor(sample_img)

plt.imshow(img_np)
plt.title(f"Label: {sample_y}")
plt.axis("off")
plt.show()

**XAI 1: Occlusion Sensitivity**

In [ ]:
def occlusion_sensitivity(model, img_tensor, patch_size=24, stride=16):
    model.eval()

    img_input = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        base_probs = torch.softmax(model(img_input), dim=1)
        pred_class = base_probs.argmax(1).item()
        base_score = base_probs[0, pred_class].item()

    _, h, w = img_tensor.shape
    heatmap = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)

    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            occluded = img_tensor.clone()
            occluded[:, y:y + patch_size, x:x + patch_size] = 0

            with torch.no_grad():
                prob = torch.softmax(
                    model(occluded.unsqueeze(0).to(DEVICE)),
                    dim=1
                )[0, pred_class].item()

            drop = base_score - prob
            heatmap[y:y + patch_size, x:x + patch_size] += drop
            counts[y:y + patch_size, x:x + patch_size] += 1

    heatmap = heatmap / np.maximum(counts, 1)
    heatmap = np.maximum(heatmap, 0)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    return heatmap

occ_map = occlusion_sensitivity(vit_model, sample_img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(occ_map, alpha=0.45)
plt.axis("off")
plt.title("Occlusion Sensitivity")
plt.tight_layout()
plt.savefig(OCCLUSION_SENSITIVITY_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 2: Integrated Gradients**

In [ ]:
def integrated_gradients_visual(model, img_tensor):
    model.eval()

    img_input = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_class = model(img_input).argmax(1).item()

    def forward_func(x):
        return model(x)

    ig = IntegratedGradients(forward_func)

    attr = ig.attribute(
        img_input,
        target=pred_class,
        n_steps=32
    )

    attr = attr.squeeze().detach().cpu().numpy()
    attr = np.abs(attr).mean(axis=0)
    attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)

    return attr

ig_map = integrated_gradients_visual(vit_model, sample_img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(ig_map, alpha=0.45)
plt.axis("off")
plt.title("Integrated Gradients")
plt.tight_layout()
plt.savefig(INTEGRATED_GRADIENTS_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 3: SmoothGrad**

In [ ]:
def smoothgrad_visual(model, img_tensor):
    model.eval()

    img_input = img_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_class = model(img_input).argmax(1).item()

    def forward_func(x):
        return model(x)

    saliency = Saliency(forward_func)
    nt = NoiseTunnel(saliency)

    attr = nt.attribute(
        img_input,
        target=pred_class,
        nt_type="smoothgrad",
        nt_samples=20,
        stdevs=0.15
    )

    attr = attr.squeeze().detach().cpu().numpy()
    attr = np.abs(attr).mean(axis=0)
    attr = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)

    return attr

sg_map = smoothgrad_visual(vit_model, sample_img)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis("off")
plt.title("Original")

plt.subplot(1, 2, 2)
plt.imshow(img_np)
plt.imshow(sg_map, alpha=0.45)
plt.axis("off")
plt.title("SmoothGrad")
plt.tight_layout()
plt.savefig(SMOOTHGRAD_PATH, dpi=300, bbox_inches="tight")
plt.show()

**XAI 4: LIME Image**

In [ ]:
def lime_predict(images_np):
    model = vit_model
    model.eval()

    batch = []
    for im in images_np:
        im = Image.fromarray((im * 255).astype(np.uint8))
        batch.append(test_tfms(im))

    batch = torch.stack(batch).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(batch), dim=1).cpu().numpy()

    return probs

explainer = lime_image.LimeImageExplainer()

explanation = explainer.explain_instance(
    img_np,
    lime_predict,
    top_labels=1,
    hide_color=0,
    num_samples=500
)

temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=8,
    hide_rest=False
)

plt.figure(figsize=(6, 6))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation")
plt.axis("off")
plt.tight_layout()
plt.savefig(LIME_PATH, dpi=300, bbox_inches="tight")
plt.show()